# 🤖 Notebook 04 — Model Experiments
**Step 7** | `notebooks/04_modeling.ipynb`

Goals:
- Try 10+ models (Logistic Regression, Random Forest, GBM …)
- Use MLflow to track experiments
- Compare models using defined success metrics
- Select the best-performing model
- Tune hyperparameters (Grid Search, Optuna, etc.)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               ExtraTreesRegressor, AdaBoostRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, mean_absolute_percentage_error)

os.makedirs('../models', exist_ok=True)
os.makedirs('../mlruns', exist_ok=True)
%matplotlib inline

## 1. Load Feature-Engineered Data

In [ ]:
try:
    df = pd.read_csv('../data/processed/delivery_features.csv')
    print('Loaded from file')
except FileNotFoundError:
    # Rebuild inline if file not present
    from sklearn.preprocessing import MinMaxScaler
    data = {
        'Distance_km': [5,22,7.93,16.42,19.52,17.44,19.03],
        'Weather': ['Clear','Rainy','Windy','Clear','Foggy','Rainy','Clear'],
        'Traffic_Level': ['Low','Medium','Low','Medium','Low','Medium','Low'],
        'Time_of_Day': ['Morning','Evening','Afternoon','Evening','Night','Afternoon','Morning'],
        'Vehicle_Type': ['Bike','Scooter','Scooter','Bike','Scooter','Scooter','Bike'],
        'Preparation_Time_min': [10,25,12,20,28,5,16],
        'Courier_Experience_yrs': [3.0,1.5,1.0,2.0,1.0,1.0,5.0],
        'Delivery_Time_min': [25,55,43,84,59,36,68],
    }
    df = pd.DataFrame(data)
    traffic_map = {'Low':1.0,'Medium':1.4,'High':1.8}
    weather_map = {'Clear':1.0,'Windy':1.1,'Foggy':1.2,'Rainy':1.3}
    df['Distance_Traffic'] = df['Distance_km'] * df['Traffic_Level'].map(traffic_map)
    df['Weather_Factor']   = df['Weather'].map(weather_map)
    df['Total_Delay_Est']  = df['Distance_Traffic'] * df['Weather_Factor'] + df['Preparation_Time_min']
    df['Experience_Level'] = pd.cut(df['Courier_Experience_yrs'],
                                     bins=[-0.1,1.0,3.0,100],
                                     labels=['Junior','Mid','Senior'])
    cat_cols = ['Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Experience_Level']
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    num_cols = ['Distance_km','Preparation_Time_min','Courier_Experience_yrs',
                'Distance_Traffic','Weather_Factor','Total_Delay_Est']
    sc = MinMaxScaler()
    df[num_cols] = sc.fit_transform(df[num_cols])
    print('Built inline')

TARGET = 'Delivery_Time_min'
X = df.drop(columns=[TARGET])
y = df[TARGET]
print('X shape:', X.shape, '| y shape:', y.shape)
df.head()

## 2. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows')

## 3. Try 10+ Models

In [ ]:
models = {
    'Linear Regression':       LinearRegression(),
    'Ridge':                   Ridge(alpha=1.0),
    'Lasso':                   Lasso(alpha=0.1),
    'ElasticNet':              ElasticNet(alpha=0.1, l1_ratio=0.5),
    'Decision Tree':           DecisionTreeRegressor(random_state=42),
    'Random Forest':           RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':       GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Extra Trees':             ExtraTreesRegressor(n_estimators=100, random_state=42),
    'AdaBoost':                AdaBoostRegressor(n_estimators=100, random_state=42),
    'SVR':                     SVR(kernel='rbf', C=10),
    'KNN':                     KNeighborsRegressor(n_neighbors=3),
}

results = {}
print(f'{'Model':<25} {'MAE':>8} {'RMSE':>8} {'R2':>8} {'MAPE%':>8}')
print('-' * 62)

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    mape = mean_absolute_percentage_error(y_test, preds) * 100

    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape, 'model': model}
    print(f'{name:<25} {mae:>8.2f} {rmse:>8.2f} {r2:>8.4f} {mape:>7.2f}%')

## 4. Visualise Model Comparison

In [ ]:
res_df = pd.DataFrame({k: {m: v for m, v in vv.items() if m != 'model'}
                       for k, vv in results.items()}).T.sort_values('MAE')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

res_df['MAE'].plot(kind='barh', ax=axes[0], color='#e74c3c', edgecolor='white')
axes[0].set_title('MAE (lower = better)')
axes[0].set_xlabel('MAE (min)')
axes[0].axvline(5, color='green', linestyle='--', label='Target ≤5')
axes[0].legend()

res_df['RMSE'].plot(kind='barh', ax=axes[1], color='#e67e22', edgecolor='white')
axes[1].set_title('RMSE (lower = better)')
axes[1].set_xlabel('RMSE (min)')

res_df['R2'].plot(kind='barh', ax=axes[2], color='#2ecc71', edgecolor='white')
axes[2].set_title('R² (higher = better)')
axes[2].set_xlabel('R²')
axes[2].axvline(0.8, color='navy', linestyle='--', label='Target ≥0.8')
axes[2].legend()

plt.suptitle('Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Cross-Validation on Top 3 Models

In [ ]:
top3 = res_df.head(3).index.tolist()
kf = KFold(n_splits=3, shuffle=True, random_state=42)

print('Cross-Validation (3-fold) — neg MAE:')
print(f'{'Model':<25}  Mean MAE   Std')
print('-' * 45)
for name in top3:
    model = results[name]['model']
    cv_scores = cross_val_score(model, X, y, cv=kf,
                                scoring='neg_mean_absolute_error')
    mean_mae = -cv_scores.mean()
    std_mae  = cv_scores.std()
    print(f'{name:<25}  {mean_mae:.2f}      ±{std_mae:.2f}')

## 6. Select the Best Model

In [ ]:
best_name = res_df['MAE'].idxmin()
best_model = results[best_name]['model']
best_metrics = {k: v for k, v in results[best_name].items() if k != 'model'}

print(f'🏆 Best model: {best_name}')
print(f'   MAE  = {best_metrics["MAE"]:.2f} min')
print(f'   RMSE = {best_metrics["RMSE"]:.2f} min')
print(f'   R²   = {best_metrics["R2"]:.4f}')
print(f'   MAPE = {best_metrics["MAPE"]:.2f} %')

## 7. Hyperparameter Tuning — GridSearchCV

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

param_grid = {
    'n_estimators':  [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth':     [2, 3, 5],
}

gs = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=0,
)
gs.fit(X_train, y_train)

print('Best params:', gs.best_params_)
tuned_preds = gs.best_estimator_.predict(X_test)
tuned_mae   = mean_absolute_error(y_test, tuned_preds)
tuned_r2    = r2_score(y_test, tuned_preds)
print(f'Tuned MAE:  {tuned_mae:.2f} min')
print(f'Tuned R²:   {tuned_r2:.4f}')

## 8. Prediction vs Actual Plot (Best Tuned Model)

In [ ]:
# Use full data for this plot since test set is tiny
final_model = gs.best_estimator_
all_preds = final_model.predict(X)

plt.figure(figsize=(7, 5))
plt.scatter(y, all_preds, color='steelblue', s=80, edgecolors='white', label='Predictions')
min_val = min(y.min(), all_preds.min()) - 5
max_val = max(y.max(), all_preds.max()) + 5
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect prediction')
plt.xlabel('Actual Delivery Time (min)')
plt.ylabel('Predicted Delivery Time (min)')
plt.title('Actual vs Predicted — Tuned GradientBoosting')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
importances = pd.Series(
    final_model.feature_importances_, index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
importances.plot(kind='bar', color='#9b59b6', edgecolor='white')
plt.title('Feature Importances — Tuned GradientBoosting')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

print('Top 5 features:')
print(importances.head())

## 10. Residuals Analysis

In [ ]:
residuals = y - all_preds

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(all_preds, residuals, color='coral', edgecolors='white', s=70)
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=8, color='coral', edgecolor='white')
axes[1].set_xlabel('Residual (min)')
axes[1].set_title('Residual Distribution')

plt.suptitle('Residuals Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Mean residual: {residuals.mean():.3f}')
print(f'Std residual:  {residuals.std():.3f}')

## 11. Log Experiments + Save Best Model

In [ ]:
# Log all experiment results to JSON (simulates MLflow tracking)
log = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'best_model': best_name,
    'tuned_params': gs.best_params_,
    'metrics': {
        name: {k: round(v, 4) for k, v in vals.items() if k != 'model'}
        for name, vals in results.items()
    },
    'tuned_metrics': {
        'MAE': round(tuned_mae, 4),
        'R2':  round(tuned_r2,  4),
    }
}

with open('../mlruns/experiment_results.json', 'w') as f:
    json.dump(log, f, indent=2)
print('✅ Experiment results logged → mlruns/experiment_results.json')

# Save best model
joblib.dump(final_model, '../models/delivery_model.pkl')
with open('../models/feature_names.json', 'w') as f:
    json.dump(list(X.columns), f)

print('✅ Model saved     → models/delivery_model.pkl')
print('✅ Feature names   → models/feature_names.json')
print(f'\n🏆 Final model: Tuned GradientBoosting')
print(f'   MAE  = {tuned_mae:.2f} min')
print(f'   R²   = {tuned_r2:.4f}')